## Neural Data Science
## Excersise 01: Encoding in single neurons
<br><br>
Jonathan Gant, Arash Shahidi, Wiktor Młynarski<br>
*LMU Biology*<br>
mlynarski@bio.lmu.de

**Excersise 1) Simulation and inference of a linear Gaussian neuron model**
<br>
<br>

1) Generate a model of a neuron with a Gabor filter (use the provided Gabor function)
<br>
2) Generate two kinds of stimuli:
<br>
<br>
a) Natural image patches - use the provided code to sample them. Standarize image patches by substracting the mean of each pixel and dividing by it's standar deviation (Z-scoring).
<br>
b) Gaussian white noise stimuli 
<br>
For each kind of stimulus simulate a neural response by taking a dot product of the filter and stimulus and adding Gaussian noise with a defined $\sigma_{noise}$
<br>
<br>
3) Compute the weighted spike-triggered average (STA) using natural stimulus / white noise responses. Visualize them alongside the original filter. How do they look? How do estimates depend on the number of stimuli you used?
<br>
<br>
4) (This is an additional, extra assignment) Compute the covariance matrix of natural stimuli, and invert it numerically. Visualize the covariance matrix, the inversion and their product. Do they look like they should? Why? Why not? Whiten the STA obtained with natural stimuli using the inverted covariant matrix and visualize alongside the non-whitened STA and the ground truth filter. Did you improve the estimate? If yes/no then why/why not?
<br>
<br>
5) Implement estimation of the receptive field via gradient-ascent maximization of the log-posterior (Maximum-A-Posteriori, MAP learning) in the Gaussian, linear model. Add an $L2$ regularization. Does this inference recover the ground truth receptive field? How does it depend on the number of data samples? How does the strength of regularization change the estimation?


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

import cv2

# define the Gabor filter function
def gabor_filter(filt_size, sigma_norm, theta, wavelength, phase=0.0,  gamma=1.0):
    """
    This function generates a gabor filter.
    Inputs:
        filt_size: size of the filter
        sigma_norm: standard deviation of the Gaussian envelope, relative to the filter size
        theta: orientation of the filter (in degrees)
        wavelength: wavelength of the sinusoidal factor
        gamma: aspect ratio of the Gaussian envelope (1.0 by default)
        phase: phase offset of the sinusoidal factor (in degrees; 0 by default)
    Outputs:
        2d matrix which defines the gabor filter. Note that this implementation pads the filter with zeros.
    """
    sigma = filt_size * sigma_norm
    
    x, y = np.meshgrid(
        np.arange(filt_size) - filt_size // 2,
        np.arange(filt_size) - filt_size // 2
    )
    x_prime = x * np.cos(np.pi * theta / 180) + y * np.sin(np.pi * theta / 180)
    y_prime = -x * np.sin(np.pi * theta / 180) + y * np.cos(np.pi * theta / 180)
    filter = np.exp(-0.5 * (x_prime**2 + (gamma * y_prime) ** 2) / sigma**2) * np.cos(
        2 * np.pi * x_prime / wavelength + np.pi * phase / 180)
    filter /= np.linalg.norm(filter)
    return filter



In [ ]:
# --- Load image and sample natural stimulus data ---

imagepath = 'landscape.png'
img = cv2.imread(imagepath)
img = np.mean(img, axis=2).astype('double')
y_max, x_max = img.shape

#standarize image
img -= np.mean(img)
img /= np.std(img)

plt.imshow(img, cmap='gray')

#parameters of samples
n_x = 20
N = 50000

#allocate the data matrix
X_natural = np.zeros((n_x**2, N))

#sample image patches
for n in range(0, N):
    #draw a random position
    x_n = np.random.randint(x_max - 2*n_x) + n_x
    y_n = np.random.randint(y_max - 2*n_x) + n_x
    
    tmp = img[y_n:y_n+n_x, x_n:x_n+n_x]
    X_natural[:, n] = tmp.flatten()

#add weak noise to avoid zeros
X_natural += np.random.randn(n_x**2, N) * 1e-6
    
#z-score image patches
X_natural -= np.mean(X_natural, axis=0)
X_natural /= np.std(X_natural, axis=0)

#plot first 25 input images
N_plot = 25
N_plot_sqrt = np.sqrt(N_plot).astype('int')
plt.figure(figsize=(20,20))
for n in range(0, N_plot):
    plt.subplot(N_plot_sqrt, N_plot_sqrt, n+1)
    plt.imshow(X_natural[:, n].reshape(n_x, n_x), cmap='gray')
    plt.axis('off')


# Excersise 2) Inference of the Poisson GLM with retinal data

a) Using the 1D retinal data from Chichnilnisky - bin it and plot spike counts
<br>
b) Implement Poisson glm fitting by maximizing log-likelihood. Use the ready-made implementation in the statsmodel package
<br>
c) Estimate temporal spike filter
<br>
d) Predict spiking on withheld data

 **Citation**: notebook adapted from [Pillow et al, *Nature* 2008](http://pillowlab.princeton.edu/pubs/abs_Pillow08_nature.html)
 
 **Data**: from [Uzzell & Chichilnisky, 2004](http://jn.physiology.org/content/92/2/780.long); see `README.txt` file in the `/data_RGCs` directory for details).
The dataset can be downloaded [here](https://pillowlab.princeton.edu/data/data_RGCs.zip):

The dataset is provided for tutorial purposes only, and should not be
distributed or used for publication without express permission from EJ
Chichilnisky (ej@stanford.edu).

## a) Plot and bin spike counts

In [ ]:
# import statements
import numpy as np
import matplotlib.pyplot as plt
from scipy.io import loadmat

# load in the data from Chichnilnisky
datadir = '../data/data_RGCs/'
stim = np.squeeze(loadmat(f'{datadir}Stim.mat')['Stim']) # contains stimulus value at each frame
stim_times = np.squeeze(loadmat(f'{datadir}stimtimes.mat')['stimtimes']) # contains time in seconds at each frame (120 Hz)
all_spike_times = [np.squeeze(x) for x in np.squeeze(loadmat(f'{datadir}SpTimes.mat')['SpTimes'])] # time of spikes for 4 neurons (in units of stim frames)

print(f'Length of stimulus: {stim.shape}')
print(f'Number of spikes for each of 4 neurons: {" ".join([str(x.size) for x in all_spike_times])}')

In [ ]:
# for an example cell plot the raw stimulus and spike times for the first second
cell_idx = 2
spike_times = all_spike_times[cell_idx]

# Print out some basic info
dt_stim = stim_times[1] - stim_times[0] # time bin size
refresh_rate = 1/dt_stim # refresh rate of the monitor
num_samples = stim.size # number of time bins in stimulus
num_spikes = spike_times.size # number of spikes

In [ ]:
# chunk the spike train into bins


In [ ]:
# plot the binned spike counts


## b) Poisson GLM

In [ ]:
# implement a GLM to predict the spike train from the stimulus
# create the design matrix


In [ ]:
import statsmodels.api as sm

# split the design matrix into training and testing sets

# define the glm model


## c) Estimate spike filter

In [ ]:
### Make plots showing filter and spike rate predictions


## d) Predict spiking on withheld data

In [ ]:
# Make plots showing and spike rate predictions
